# Unreal Engine 5 Documentation Scraper

Scrapes the official UE5 documentation from `dev.epicgames.com` for versions 5.0 through 5.7.
Outputs structured JSONL files suitable for building a RAG system with Gemini APIs.

**Strategy:**
1. Use Playwright (headless Chromium) to handle Cloudflare protection and JS rendering
2. Discover all documentation page URLs via the sidebar navigation tree
3. Scrape each page's content, extracting clean text with section structure
4. Save incrementally as JSONL, with full resumability

In [13]:
%pip install playwright beautifulsoup4 tqdm lxml

Note: you may need to restart the kernel to use updated packages.


In [14]:
import subprocess
subprocess.run(["playwright", "install", "chromium"], check=True)
print("Chromium installed successfully.")

Chromium installed successfully.


## Imports & Configuration

In [28]:
import asyncio
import json
import os
import re
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse
from collections import defaultdict

from bs4 import BeautifulSoup, NavigableString
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout

BASE_URL = "https://dev.epicgames.com/documentation/en-us/unreal-engine"
# For full multi-version crawls, list every version you want. For hub-only (5.7 section docs), use ["5.7"].
VERSIONS = ["5.7"]

# "hub" = only crawl docs reachable from the official 5.7 landing page section cards (What's New, Basics, …)
# "full" = legacy behavior: seed from the version landing page + root (broader / slower)
DISCOVERY_MODE = "hub"  # "hub" | "full"
HUB_VERSION = "5.7"  # must match VERSIONS when using hub mode

# Full left-hand contents table (same order as the site TOC). Child pages are discovered by BFS from these roots.
# "Get Started" in the UI is a label only (no href) — entry point is "Understanding the Basics" / landing.
UE57_DOCUMENTATION_HUB_SEEDS = [
    f"{BASE_URL}/unreal-engine-5-7-documentation",
    f"{BASE_URL}/whats-new",
    f"{BASE_URL}/understanding-the-basics-of-unreal-engine",
    f"{BASE_URL}/working-with-content-in-unreal-engine",
    f"{BASE_URL}/building-virtual-worlds-in-unreal-engine",
    f"{BASE_URL}/designing-visuals-rendering-and-graphics-with-unreal-engine",
    f"{BASE_URL}/creating-visual-effects-in-niagara-for-unreal-engine",
    f"{BASE_URL}/gameplay-tutorials-for-unreal-engine",
    f"{BASE_URL}/blueprints-visual-scripting-in-unreal-engine",
    f"{BASE_URL}/programming-with-cplusplus-in-unreal-engine",
    f"{BASE_URL}/gameplay-systems-in-unreal-engine",
    f"{BASE_URL}/getting-started-with-mobile-development-in-unreal-engine",
    f"{BASE_URL}/animating-characters-and-objects-in-unreal-engine",
    f"{BASE_URL}/motion-design-in-unreal-engine",
    f"{BASE_URL}/creating-user-interfaces-with-umg-and-slate-in-unreal-engine",
    f"{BASE_URL}/working-with-audio-in-unreal-engine",
    f"{BASE_URL}/working-with-media-in-unreal-engine",
    f"{BASE_URL}/setting-up-your-production-pipeline-in-unreal-engine",
    f"{BASE_URL}/testing-and-optimizing-your-content",
    f"{BASE_URL}/sharing-and-releasing-projects-for-unreal-engine",
    f"{BASE_URL}/samples-and-tutorials-for-unreal-engine",
    f"{BASE_URL}/WebAPI",
    f"{BASE_URL}/BlueprintAPI",
    f"{BASE_URL}/API",
    f"{BASE_URL}/PythonAPI",
    f"{BASE_URL}/node-reference",
]

DISCOVERY_MAX_PAGES_HUB = 15_000
DISCOVERY_MAX_PAGES_FULL = 300

OUTPUT_DIR = Path("scraped_data")
URLS_DIR = OUTPUT_DIR / "urls"
CONTENT_DIR = OUTPUT_DIR / "content"

OUTPUT_DIR.mkdir(exist_ok=True)
URLS_DIR.mkdir(exist_ok=True)
CONTENT_DIR.mkdir(exist_ok=True)

REQUEST_DELAY = 1.5  # seconds between page loads when scraping content (be polite to Epic's servers)
NAV_TIMEOUT = 30_000  # ms to wait for page navigation
CONTENT_TIMEOUT = 15_000  # ms to wait for content elements
# Discovery is I/O-bound; long fixed sleeps + expanding every nav button made it ~15–25s/page.
DISCOVERY_SETTLE_MS = 500  # ms after domcontentloaded during URL discovery (was 2000)
DISCOVERY_BETWEEN_PAGES_S = 0.15  # light pacing during discovery only
DISCOVERY_SIDEBAR_MAX_CLICKS = 120  # cap expand clicks per page (sidebar was the main slowdown)
DISCOVERY_SIDEBAR_CLICK_DELAY_MS = 40
SCRAPE_EXTRA_WAIT_MS = 2000  # after load, let React render doc body before reading HTML
MIN_DOC_CHARS = 25  # below this, treat page as failed/empty

# True = follow API/Blueprint/Python/Web hubs from the TOC (huge; may need more RAM / browser restarts)
INCLUDE_API_REFERENCE = True

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Versions to scrape: {VERSIONS}")
print(f"Discovery mode: {DISCOVERY_MODE} (hub seeds for UE {HUB_VERSION})" if DISCOVERY_MODE == "hub" else f"Discovery mode: {DISCOVERY_MODE}")
print(f"Include API reference: {INCLUDE_API_REFERENCE}")

Output directory: /Users/saanvitaneja/Projects/llm_engineering/Unreal Engine Scraper/scraped_data
Versions to scrape: ['5.7']
Discovery mode: hub (hub seeds for UE 5.7)
Include API reference: False


## Helper Functions

In [27]:
API_PATH_PREFIXES = ("/API/", "/BlueprintAPI/", "/PythonAPI/", "/WebAPI/", "/python-api/")


def normalize_doc_url(url: str) -> str | None:
    """Extract and normalize a documentation URL, stripping fragments and non-doc params."""
    parsed = urlparse(url)
    if "dev.epicgames.com" not in parsed.netloc and parsed.netloc != "":
        return None
    path = parsed.path
    if not path.startswith("/documentation/en-us/unreal-engine"):
        return None
    # Keep only application_version param if present
    qs = parse_qs(parsed.query)
    new_qs = {}
    if "application_version" in qs:
        new_qs["application_version"] = qs["application_version"][0]
    query_str = urlencode(new_qs) if new_qs else ""
    return urlunparse(("https", "dev.epicgames.com", path, "", query_str, ""))


def is_api_reference(path: str) -> bool:
    """Check if a URL path points to auto-generated API reference docs."""
    return any(prefix in path for prefix in API_PATH_PREFIXES)


def slug_from_url(url: str) -> str:
    """Create a filesystem-safe identifier from a URL."""
    parsed = urlparse(url)
    key = parsed.path + ("?" + parsed.query if parsed.query else "")
    return hashlib.md5(key.encode()).hexdigest()


def load_url_set(filepath: Path) -> set[str]:
    """Load a set of URLs from a newline-delimited file."""
    if not filepath.exists():
        return set()
    return set(filepath.read_text().strip().splitlines())


def save_url_set(urls: set[str], filepath: Path):
    """Save a set of URLs to a newline-delimited file."""
    filepath.write_text("\n".join(sorted(urls)) + "\n")


def load_scraped_urls(version: str) -> set[str]:
    """Return URLs already scraped for a given version by reading the JSONL."""
    jsonl_path = CONTENT_DIR / f"ue_{version.replace('.', '_')}.jsonl"
    scraped = set()
    if jsonl_path.exists():
        with open(jsonl_path) as f:
            for line in f:
                try:
                    scraped.add(json.loads(line)["url"])
                except (json.JSONDecodeError, KeyError):
                    continue
    return scraped


def clean_text(text: str) -> str:
    """Collapse whitespace and strip a text string."""
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def url_with_application_version(url: str, version: str) -> str:
    """Force query ?application_version=… to the version being scraped.

    Discovery caches links from pages that point at mixed engine versions; without this,
    you request 5.2/5.3 hubs while 'scraping' 5.0 and extraction often fails."""
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    qs["application_version"] = [version]
    flat = {k: v[0] for k, v in qs.items()}
    new_query = urlencode(flat)
    return urlunparse(("https", parsed.netloc, parsed.path, "", new_query, ""))


def canonical_doc_url_for_version(url: str, version: str) -> str | None:
    """Single form per doc page for this crawl: normalized path + target application_version.

    Used so the same page is not queued twice because links used different query strings."""
    n = normalize_doc_url(url)
    if n is None:
        return None
    return url_with_application_version(n, version)


def parse_epic_contents_table_html(
    html: str,
    *,
    base: str = "https://dev.epicgames.com",
) -> list[dict[str, str]]:
    """Parse Epic Dev Portal TOC HTML (``contents-table-*``): each ``a.contents-table-link`` with ``href``.

    Returns ``{"title", "url"}`` dicts in document order. Label-only rows (e.g. ``<span>`` with no ``<a>``)
    are skipped. Duplicate ``href`` values are dropped (first wins).
    """
    soup = BeautifulSoup(html, "lxml")
    seen: set[str] = set()
    out: list[dict[str, str]] = []
    for a in soup.select("a.contents-table-link[href]"):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        abs_url = urljoin(base, href)
        if abs_url in seen:
            continue
        seen.add(abs_url)
        title = clean_text(a.get_text())
        if not title:
            title = abs_url.rstrip("/").split("/")[-1] or abs_url
        out.append({"title": title, "url": abs_url})
    return out


def contents_table_urls_for_scraping(entries: list[dict[str, str]], version: str) -> list[str]:
    """Turn TOC parse rows into normalized scrape URLs with ``application_version`` set."""
    urls: list[str] = []
    seen: set[str] = set()
    for row in entries:
        n = normalize_doc_url(row["url"])
        if n is None:
            continue
        u = url_with_application_version(n, version)
        if u in seen:
            continue
        seen.add(u)
        urls.append(u)
    return urls


def save_contents_table_jsonl(entries: list[dict[str, str]], filepath: Path) -> None:
    """Persist TOC rows as JSONL (``title``, ``url`` per line)."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        for row in entries:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

## UE Doc Scraper Class

Core scraper that uses Playwright to:
1. **Discover** all documentation page URLs by walking the sidebar navigation tree
2. **Scrape** each page's content into structured text with section breakdowns
3. **Save** results incrementally as JSONL for RAG ingestion

## Phase 1: Discover Documentation URLs

**Hub mode:** walks from the 5.7 landing section cards, follows in-doc links, caches to `urls/urls_hub_5_7.txt`.  
**Full mode:** narrower legacy seeds, caches to `urls/urls_{version}.txt`. Re-run discovery if you change mode or seeds.

In [29]:
scraper = UEDocScraper(headless=True)
await scraper.start()

all_urls: dict[str, set[str]] = {}

for version in VERSIONS:
    print(f"\n{'='*60}")
    print(f"Discovering URLs for Unreal Engine {version}")
    print(f"{'='*60}")
    urls = await scraper.discover_urls(version)
    all_urls[version] = urls
    print(f"  Total unique URLs for v{version}: {len(urls)}")

print(f"\n{'='*60}")
print("Discovery Summary")
print(f"{'='*60}")
for v, urls in all_urls.items():
    print(f"  v{v}: {len(urls)} pages")
total = sum(len(u) for u in all_urls.values())
print(f"  Total across all versions: {total}")

Browser started.

Discovering URLs for Unreal Engine 5.7
  Hub mode: 21 seeds from UE 5.7 documentation landing page
  Starting URL discovery for v5.7 (max 8000 pages visited)...


Discovering v5.7:   0%|          | 0/8000 [00:00<?, ?it/s]

Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver
Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


  Discovered 28 URLs for v5.7
  Total unique URLs for v5.7: 28

Discovery Summary
  v5.7: 28 pages
  Total across all versions: 28


## Phase 2: Scrape Page Content

Fetches and extracts content from every discovered URL. Progress is saved incrementally
to JSONL files -- if interrupted, re-running this cell will resume where it left off.

In [31]:
# Load URLs from cache if discovery cell was run in a prior session
try:
    all_urls
except NameError:
    all_urls = {}

if not all_urls:
    for version in VERSIONS:
        cache_file = URLS_DIR / f"urls_{version.replace('.', '_')}.txt"
        all_urls[version] = load_url_set(cache_file)

for version in VERSIONS:
    urls = all_urls.get(version, set())
    if not urls:
        print(f"No URLs found for v{version}, skipping.")
        continue
    print(f"\n{'='*60}")
    print(f"Scraping Unreal Engine {version} ({len(urls)} pages)")
    print(f"{'='*60}")
    await scraper.scrape_version(version, urls)

print("\nScraping complete.")


Scraping Unreal Engine 5.7 (28 pages)
  Scraping 28 pages for v5.7 (0 already done)


Scraping v5.7:   0%|          | 0/28 [00:00<?, ?it/s]

  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/API?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 2s
  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/API?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 4s
  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/API?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 8s
  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/BlueprintAPI?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 2s
  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/BlueprintAPI?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 4s


pipe closed by peer or os.write(pipe, data) raised exception.


  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/BlueprintAPI?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 8s


...

pipe closed by peer or os.write(pipe, data) raised exception.


  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/PythonAPI?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 8s


pipe closed by peer or os.write(pipe, data) raised exception.


  Error on https://dev.epicgames.com/documentation/en-us/unreal-engine/WebAPI?application_version=5.7: Page.goto: Connection closed while reading from the driver, retrying in 4s


CancelledError: 

In [ ]:
await scraper.stop()

## Phase 3: Inspect & Export Results

View statistics and sample data from the scraped content.

In [ ]:
print("Scraped Data Summary")
print("=" * 60)

total_pages = 0
total_chars = 0

for version in VERSIONS:
    jsonl_path = CONTENT_DIR / f"ue_{version.replace('.', '_')}.jsonl"
    if not jsonl_path.exists():
        print(f"  v{version}: no data")
        continue
    pages = 0
    chars = 0
    with open(jsonl_path) as f:
        for line in f:
            try:
                doc = json.loads(line)
                pages += 1
                chars += len(doc.get("content", ""))
            except json.JSONDecodeError:
                continue
    total_pages += pages
    total_chars += chars
    print(f"  v{version}: {pages:>5} pages, {chars:>10,} chars")

print(f"\n  Total: {total_pages} pages, {total_chars:,} characters")
print(f"  Output directory: {CONTENT_DIR.resolve()}")

In [ ]:
# Preview a sample document
sample_version = "5.7"
jsonl_path = CONTENT_DIR / f"ue_{sample_version.replace('.', '_')}.jsonl"

if jsonl_path.exists():
    with open(jsonl_path) as f:
        first_line = f.readline()
        if first_line:
            doc = json.loads(first_line)
            print(f"Title: {doc['title']}")
            print(f"URL: {doc['url']}")
            print(f"Version: {doc['version']}")
            print(f"Breadcrumb: {' > '.join(doc.get('breadcrumb', []))}")
            print(f"Sections: {len(doc.get('sections', []))}")
            print(f"Content length: {len(doc['content']):,} chars")
            print(f"\nFirst 500 chars of content:")
            print("-" * 40)
            print(doc["content"][:500])
else:
    print(f"No data for v{sample_version} yet. Run the scraper first.")

## Bonus: Scrape UnrealSpecifiers from GitHub

The [UnrealSpecifiers](https://github.com/fjz13/UnrealSpecifiers) repo contains detailed docs for
100+ specifiers and 300+ meta tags (UCLASS, UPROPERTY, UFUNCTION, etc.) that the official docs
barely cover. Since these are plain markdown files on GitHub, we fetch them directly via the
GitHub API -- no browser needed.

In [21]:
import sys

import requests

SPECIFIERS_REPO = "fjz13/UnrealSpecifiers"
SPECIFIERS_BRANCH = "main"
SPECIFIERS_DOC_PREFIX = "Doc/en/"
SPECIFIERS_OUTPUT = CONTENT_DIR / "unreal_specifiers.jsonl"
# Print a heartbeat every N successful files so you see progress even if tqdm lags in Jupyter
SPECIFIERS_LOG_EVERY = 25

_SESSION = requests.Session()
_SESSION.headers.update(
    {
        "User-Agent": "UnrealEngine-ScraperNotebook/1.0 (educational; contact: local)",
        "Accept": "application/vnd.github+json",
    }
)
_gh = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
if _gh:
    _SESSION.headers["Authorization"] = f"token {_gh}"


def _log(msg: str) -> None:
    print(msg, flush=True)


def _format_err(e: BaseException) -> str:
    if isinstance(e, requests.HTTPError) and e.response is not None:
        r = e.response
        body = (r.text or "").replace("\n", " ")[:400]
        return f"HTTP {r.status_code} {r.reason!r} url={r.url!r} body[:400]={body!r}"
    return f"{type(e).__name__}: {e}"


def fetch_github_tree(repo: str, branch: str) -> list[dict]:
    """Get the full recursive file tree from a GitHub repo."""
    url = f"https://api.github.com/repos/{repo}/git/trees/{branch}?recursive=1"
    _log(f"[UnrealSpecifiers] GET tree {url}")
    resp = _SESSION.get(url, timeout=60)
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        _log(f"[UnrealSpecifiers] FATAL tree request failed: {_format_err(e)}")
        raise
    data = resp.json()
    tree = data.get("tree", [])
    _log(f"[UnrealSpecifiers] tree OK: {len(tree)} objects in API response")
    return tree


def fetch_raw_markdown(repo: str, branch: str, path: str) -> str:
    """Fetch raw file content from GitHub."""
    url = f"https://raw.githubusercontent.com/{repo}/{branch}/{path}"
    resp = _SESSION.get(url, timeout=60)
    resp.raise_for_status()
    return resp.text


def parse_specifier_md(content: str, path: str) -> dict:
    """Parse a specifier markdown file into structured data."""
    lines = content.strip().split("\n")

    # Extract title from first heading
    title = path.split("/")[-1].replace(".md", "")
    for line in lines:
        if line.startswith("# "):
            title = line.lstrip("# ").strip()
            break

    # Determine category from path (Specifier, Meta, Flags)
    parts = path.replace(SPECIFIERS_DOC_PREFIX, "").split("/")
    category = parts[0] if parts else "General"
    subcategory = parts[1] if len(parts) > 2 else ""

    return {
        "title": title,
        "content": content,
        "category": category,
        "subcategory": subcategory,
        "source": "UnrealSpecifiers",
        "source_url": f"https://github.com/{SPECIFIERS_REPO}/blob/{SPECIFIERS_BRANCH}/{path}",
        "path": path,
    }


def scrape_unreal_specifiers():
    """Scrape all English markdown docs from the UnrealSpecifiers repo."""
    already_scraped = set()
    if SPECIFIERS_OUTPUT.exists():
        with open(SPECIFIERS_OUTPUT) as f:
            for line in f:
                try:
                    already_scraped.add(json.loads(line)["path"])
                except (json.JSONDecodeError, KeyError):
                    continue

    _log(f"[UnrealSpecifiers] output file: {SPECIFIERS_OUTPUT.resolve()}")
    _log(f"[UnrealSpecifiers] resume: {len(already_scraped)} paths already in jsonl")
    if _gh:
        _log("[UnrealSpecifiers] GITHUB_TOKEN/GH_TOKEN is set (better API rate limits)")

    tree = fetch_github_tree(SPECIFIERS_REPO, SPECIFIERS_BRANCH)
    md_files = [
        item["path"]
        for item in tree
        if item["path"].startswith(SPECIFIERS_DOC_PREFIX)
        and item["path"].endswith(".md")
        and item["type"] == "blob"
    ]
    _log(f"[UnrealSpecifiers] markdown files under {SPECIFIERS_DOC_PREFIX!r}: {len(md_files)}")

    to_scrape = [p for p in md_files if p not in already_scraped]
    if not to_scrape:
        _log(f"[UnrealSpecifiers] nothing to do — all {len(md_files)} docs already in jsonl.")
        return

    n = len(to_scrape)
    _log(f"[UnrealSpecifiers] starting: will fetch {n} files ({len(already_scraped)} skipped as done)")
    scraped = 0
    errors = 0

    with open(SPECIFIERS_OUTPUT, "a") as f:
        bar = tqdm(
            to_scrape,
            desc="UnrealSpecifiers",
            file=sys.stdout,
            mininterval=0.5,
            dynamic_ncols=True,
        )
        for path in bar:
            try:
                content = fetch_raw_markdown(SPECIFIERS_REPO, SPECIFIERS_BRANCH, path)
                doc = parse_specifier_md(content, path)
                doc["scraped_at"] = datetime.now(timezone.utc).isoformat()
                f.write(json.dumps(doc, ensure_ascii=False) + "\n")
                f.flush()
                scraped += 1
                if scraped == 1:
                    _log(f"[UnrealSpecifiers] first OK: {path} ({len(content)} chars)")
                elif scraped % SPECIFIERS_LOG_EVERY == 0:
                    _log(
                        f"[UnrealSpecifiers] progress: {scraped}/{n} saved, {errors} errors "
                        f"(last OK: {path})"
                    )
            except Exception as e:
                errors += 1
                _log(f"[UnrealSpecifiers] FAIL ({errors} total failures): {path}")
                _log(f"           -> {_format_err(e)}")
            time.sleep(0.3)

    _log(
        f"[UnrealSpecifiers] finished: saved={scraped}, errors={errors}, "
        f"output={SPECIFIERS_OUTPUT.resolve()}"
    )
    if errors:
        _log(
            "[UnrealSpecifiers] had failures — check HTTP 403 (rate limit: set GITHUB_TOKEN or wait), "
            "timeouts, or network issues."
        )


scrape_unreal_specifiers()

[UnrealSpecifiers] output file: /Users/saanvitaneja/Projects/llm_engineering/Unreal Engine Scraper/scraped_data/content/unreal_specifiers.jsonl
[UnrealSpecifiers] resume: 15 paths already in jsonl
[UnrealSpecifiers] GET tree https://api.github.com/repos/fjz13/UnrealSpecifiers/git/trees/main?recursive=1


[UnrealSpecifiers] tree OK: 3927 objects in API response
[UnrealSpecifiers] markdown files under 'Doc/en/': 606
[UnrealSpecifiers] starting: will fetch 591 files (15 skipped as done)


UnrealSpecifiers:   0%|          | 0/591 [00:00<?, ?it/s]

[UnrealSpecifiers] first OK: Doc/en/Flags/EClassFlags/CLASS_Hidden.md (166 chars)
[UnrealSpecifiers] progress: 25/591 saved, 0 errors (last OK: Doc/en/Flags/EFunctionFlags.md)
[UnrealSpecifiers] progress: 50/591 saved, 0 errors (last OK: Doc/en/Flags/EFunctionFlags/FUNC_Private.md)
[UnrealSpecifiers] progress: 75/591 saved, 0 errors (last OK: Doc/en/Flags/EPropertyFlags/CPF_Config.md)
[UnrealSpecifiers] progress: 100/591 saved, 0 errors (last OK: Doc/en/Flags/EPropertyFlags/CPF_NonPIEDuplicateTransient.md)
[UnrealSpecifiers] progress: 125/591 saved, 0 errors (last OK: Doc/en/Flags/EStructFlags/STRUCT_IdenticalNative.md)
[UnrealSpecifiers] progress: 150/591 saved, 0 errors (last OK: Doc/en/Meta/AnimationGraph/AnimGetter/AnimGetter.md)
[UnrealSpecifiers] progress: 175/591 saved, 0 errors (last OK: Doc/en/Meta/Blueprint/BlueprintThreadSafe/BlueprintThreadSafe.md)
[UnrealSpecifiers] progress: 200/591 saved, 0 errors (last OK: Doc/en/Meta/Blueprint/KismetHideOverrides/KismetHideOverrides.md

## Merge Everything into a Single File

Combines official docs (all versions) + UnrealSpecifiers into one JSONL for RAG ingestion.

In [ ]:
merged_path = OUTPUT_DIR / "ue5_docs_all_versions.jsonl"
total = 0

source_files = []
# Official docs per version
for version in VERSIONS:
    source_files.append(CONTENT_DIR / f"ue_{version.replace('.', '_')}.jsonl")
# UnrealSpecifiers
source_files.append(SPECIFIERS_OUTPUT)

with open(merged_path, "w") as out:
    for src in source_files:
        if not src.exists():
            continue
        with open(src) as f:
            for line in f:
                line = line.strip()
                if line:
                    out.write(line + "\n")
                    total += 1

print(f"Merged {total} documents into {merged_path.resolve()}")